# Exploratory Data Analysis (EDA)

Telco Customer Churn dataset – initial exploration.


In [1]:
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import os

import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("../notebooks/figures", exist_ok=True)

In [2]:
# Load processed data (generated by data pipeline)
processed_path = "../data/processed/telco_churn_processed.parquet"
df = pd.read_parquet(processed_path)
df.head()

,Churn,Contract,Dependents,DeviceProtection,InternetService,MonthlyCharges,MultipleLines,OnlineBackup,OnlineSecurity,PaperlessBilling,Partner,PaymentMethod,PhoneService,SeniorCitizen,StreamingMovies,StreamingTV,TechSupport,TotalCharges,gender,tenure
0,No,Month-to-month,No,No,DSL,29.85,No phone service,Yes,No,Yes,Yes,Electronic check,No,0,No,No,No,29.85,Female,1
1,No,One year,No,Yes,DSL,56.95,No,No,Yes,No,No,Mailed check,Yes,0,No,No,No,1889.50,Male,34
2,Yes,Month-to-month,No,No,DSL,53.85,No,Yes,Yes,Yes,No,Mailed check,Yes,0,No,No,No,108.15,Male,2
3,No,One year,No,Yes,DSL,42.30,No phone service,No,Yes,No,No,Bank transfer (automatic),No,0,No,No,Yes,1840.75,Male,45
4,Yes,Month-to-month,No,No,Fiber optic,70.70,No,No,No,Yes,No,Electronic check,Yes,0,No,No,No,151.65,Female,2


In [3]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (7043, 20)
Columns: ['Churn', 'Contract', 'Dependents', 'DeviceProtection', 'InternetService', 'MonthlyCharges', 'MultipleLines', 'OnlineBackup', 'OnlineSecurity', 'PaperlessBilling', 'Partner', 'PaymentMethod', 'PhoneService', 'SeniorCitizen', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'TotalCharges', 'gender', 'tenure']


In [4]:
# Missing values per column
missing = df.isnull().sum()
print(missing[missing > 0])

TotalCharges    11
dtype: int64


In [5]:
# Duplicate analysis
print("Full-row duplicates (raw CSV had unique customerID):", df.duplicated().sum())
# Note: After dropping customerID, some feature rows may be identical
# This is acceptable for ML - we do not drop during EDA
feature_dupes = df.duplicated().sum()
print(f"Feature duplicates (after dropping customerID): {feature_dupes}")

Full-row duplicates (raw CSV had unique customerID): 22
Feature duplicates (after dropping customerID): 22


In [6]:
# Target distribution
target_counts = df["Churn"].value_counts()
print(target_counts)
target_pct = df["Churn"].value_counts(normalize=True) * 100
print(target_pct.round(2).astype(str) + "%")
ax = target_counts.plot(kind="bar")
plt.title("Churn distribution")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.tight_layout()
fig_path = "../notebooks/figures/churn_distribution.png"
plt.savefig(fig_path, dpi=150)
print(f"Saved: {fig_path}")
plt.close()

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.46%
Yes    26.54%
Name: proportion, dtype: str


Saved: ../notebooks/figures/churn_distribution.png


In [7]:
# Numeric feature summary
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", num_cols)
print(df[num_cols].describe())

Numeric columns: ['MonthlyCharges', 'SeniorCitizen', 'TotalCharges', 'tenure']
       MonthlyCharges  SeniorCitizen  TotalCharges       tenure
count     7043.000000    7043.000000   7032.000000  7043.000000
mean        64.761692       0.162147   2283.300441    32.371149
std         30.090047       0.368612   2266.771362    24.559481
min         18.250000       0.000000     18.800000     0.000000
25%         35.500000       0.000000    401.450000     9.000000
50%         70.350000       0.000000   1397.475000    29.000000
75%         89.850000       0.000000   3794.737500    55.000000
max        118.750000       1.000000   8684.800000    72.000000


In [8]:
# Categorical feature summary
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print("Categorical columns:", cat_cols)
for col in cat_cols:
    print(f"\n=== {col} ===")
    print(f"Unique values: {df[col].nunique()}")
    vc = df[col].value_counts()
    print(vc.head(10))
    if len(vc) > 10:
        print(f"... and {len(vc) - 10} more values")

Categorical columns: ['Churn', 'Contract', 'Dependents', 'DeviceProtection', 'InternetService', 'MultipleLines', 'OnlineBackup', 'OnlineSecurity', 'PaperlessBilling', 'Partner', 'PaymentMethod', 'PhoneService', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'gender']

=== Churn ===
Unique values: 2
Churn
No     5174
Yes    1869
Name: count, dtype: int64

=== Contract ===
Unique values: 3
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

=== Dependents ===
Unique values: 2
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

=== DeviceProtection ===
Unique values: 3
DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

=== InternetService ===
Unique values: 3
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

=== MultipleLines ===
Unique values: 3
MultipleLines
No                  3390
Yes         

In [9]:
# Correlation heatmap for numeric features
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (numeric features)")
plt.tight_layout()
fig_path = "../notebooks/figures/correlation_heatmap.png"
plt.savefig(fig_path, dpi=150)
print(f"Saved: {fig_path}")
plt.close()

Saved: ../notebooks/figures/correlation_heatmap.png


In [10]:
# Churn rate by key categorical features
key_cats = ["Contract", "InternetService", "PaymentMethod", "SeniorCitizen"]
for col in key_cats:
    if col in df.columns:
        print(f"\n=== Churn rate by {col} ===")
        crosstab = pd.crosstab(df[col], df["Churn"], normalize="index") * 100
        print(crosstab.round(2))


=== Churn rate by Contract ===
Churn              No    Yes
Contract                    
Month-to-month  57.29  42.71
One year        88.73  11.27
Two year        97.17   2.83

=== Churn rate by InternetService ===
Churn               No    Yes
InternetService              
DSL              81.04  18.96
Fiber optic      58.11  41.89
No               92.60   7.40

=== Churn rate by PaymentMethod ===
Churn                         No    Yes
PaymentMethod                          
Bank transfer (automatic)  83.29  16.71
Credit card (automatic)    84.76  15.24
Electronic check           54.71  45.29
Mailed check               80.89  19.11

=== Churn rate by SeniorCitizen ===
Churn             No    Yes
SeniorCitizen              
0              76.39  23.61
1              58.32  41.68
